# 06 · Build a reproducible experiment / ကိုယ်ပိုင် experiment တည်ဆောက်ခြင်း

**Goal:** compare ideal, exact-noisy and sampled-noisy expectations with a reusable template.
About 15 minutes. Keep this notebook as a starting point for your next experiment.

မေးခွန်း၊ circuit၊ noise model၊ seed နှင့် အကဲဖြတ်မည့် metric ကို ရှင်းလင်းစွာ သတ်မှတ်ပါ။


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nexqus
from nexqus import Circuit, Simulator, Parameter, Noise, NoiseModel, algorithms
assert nexqus.__version__ == '1.4.0rc1', 'Install NexQus SDK 1.4.0rc1 in this kernel environment and restart it'
print('NexQus SDK:', nexqus.__version__)


**Question:** how does the specified depolarizing model change the first qubit's Z expectation
as the input angle varies? We compare simulation methods, not quantum versus classical computational advantage.


In [ ]:
theta = Parameter('theta')
template = Circuit(2).ry(theta, 0).cx(0, 1)
bindings = [{'theta': float(t)} for t in np.linspace(0, np.pi, 9)]
model = NoiseModel(Noise.depolarizing(.05))
sim = Simulator()
ideal = sim.run_sweep(template, bindings, seed=17, shots=0, retain_state=True)
exact = sim.run_sweep(template, bindings, noise=model, seed=17, shots=0, retain_state=True)
ideal_z = np.array([r.expectation('ZI') for r in ideal])
exact_z = np.array([r.expectation('ZI') for r in exact])
# q0 is touched by RY and CX, so local depolarization attenuates Z twice.
np.testing.assert_allclose(exact_z, .95**2 * ideal_z, atol=1e-12)


In [ ]:
trajectories = Simulator(method='trajectory').run_sweep(template, bindings, noise=model,
    seed=17, shots=0, trajectories=2048, batch_size=64)
# A diagonal observable can be evaluated from probabilities without retaining a density matrix.
sampled_z = np.array([r.marginal([0])[0] - r.marginal([0])[1] for r in trajectories])
max_error = float(np.max(np.abs(exact_z - sampled_z)))
print('Max expectation discrepancy:', max_error)
assert max_error < .1


In [ ]:
angle_values = [b['theta'] for b in bindings]
plt.figure(figsize=(7, 3))
plt.plot(angle_values, ideal_z, label='Ideal')
plt.plot(angle_values, exact_z, label='Exact specified noise')
plt.scatter(angle_values, sampled_z, label='Trajectory estimate', color='#f59e0b')
plt.xlabel('Input angle (radians)'); plt.ylabel('Expectation ZI')
plt.title('A reproducible circuit experiment'); plt.legend()
plt.tight_layout(); plt.show()


In [ ]:
import json
report = dict(sdk_version=nexqus.__version__, backend='numpy', precision='double',
              base_seed=17, noise='post-gate local depolarizing p=0.05',
              trajectories=2048, batch_size=64, max_expectation_error=max_error,
              angles=angle_values, ideal=ideal_z.tolist(), exact=exact_z.tolist(),
              sampled=sampled_z.tolist())
print(json.dumps(report, indent=2))
print('All experiment checks passed.')


## Next steps / ဆက်လက်လေ့လာရန်

1. Change one variable at a time: noise probability, circuit depth or input angles.
2. Repeat stochastic experiments with several seeds and report variability.
3. Add a new application only after defining a relevant classical baseline and metric.
4. Use `examples/train_sine.py`, `examples/vqe_two_qubits.py` and the API guides as starting points.

တစ်ကြိမ်လျှင် variable တစ်ခုသာ ပြောင်းပြီး မူလ experiment နှင့် နှိုင်းယှဉ်ပါ။
သင်ခန်းစာများ၏ passing assertions သည် ဤ example များအတွက်သာ မှန်ကန်ကြောင်း ပြပါသည်။
